In [3]:
import os
import numpy as np
import cv2
import joblib
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.decomposition import PCA
import gc

In [4]:
img_size = (64, 64)
data_dir = 'split_dataset'
n_neighbors = 7
n_splits = 5

def preprocess_image(image, target_size=(64, 64)):
    image = cv2.resize(image, target_size)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = image.astype("float32") / 255.0 
    return image

def load_images_from_folder(folder):
    X, y = [], []
    for class_name in os.listdir(folder):
        class_path = os.path.join(folder, class_name)
        if not os.path.isdir(class_path):
            continue
        for file in os.listdir(class_path):
            file_path = os.path.join(class_path, file)
            image = cv2.imread(file_path)
            if image is None:
                continue
            image = preprocess_image(image, target_size=img_size)
            X.append(image)
            y.append(class_name)
    return np.array(X), np.array(y)


In [ ]:
X_train, y_train = load_images_from_folder(os.path.join(data_dir, 'train'))
X_val, y_val = load_images_from_folder(os.path.join(data_dir, 'val'))
X_test, y_test = load_images_from_folder(os.path.join(data_dir, 'test'))

X_all = np.concatenate([X_train, X_val, X_test])
y_all = np.concatenate([y_train, y_val, y_test])
X_all_flat = X_all.reshape(len(X_all), -1)

le = LabelEncoder()
y_all_enc = le.fit_transform(y_all)


In [ ]:
print("Starting Stratified K-Fold Cross-Validation...\n")
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
all_y_true = []
all_y_pred = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X_all_flat, y_all_enc), 1):
    print(f"--- Fold {fold} ---")
    
    X_train_fold, X_test_fold = X_all_flat[train_idx], X_all_flat[test_idx]
    y_train_fold, y_test_fold = y_all_enc[train_idx], y_all_enc[test_idx]

    pca = PCA(n_components=0.95)
    X_train_pca = pca.fit_transform(X_train_fold)
    X_test_pca = pca.transform(X_test_fold)

    knn = KNeighborsClassifier(n_neighbors=n_neighbors, weights='distance')
    knn.fit(X_train_pca, y_train_fold)

    y_pred = knn.predict(X_test_pca)
    all_y_true.extend(y_test_fold)
    all_y_pred.extend(y_pred)
    print(classification_report(y_test_fold, y_pred, target_names=le.classes_))
    print()

    del X_train_fold, X_test_fold, X_train_pca, X_test_pca, knn, pca
    gc.collect()

Starting Stratified K-Fold Cross-Validation...

--- Fold 1 ---
              precision    recall  f1-score   support

           क       0.93      0.91      0.92       215
         क्ष       1.00      1.00      1.00       183
           ख       0.98      0.87      0.92       215
           ग       0.87      0.93      0.90       215
           घ       0.98      0.99      0.99       185
           ङ       0.99      0.91      0.95       182
           च       0.95      0.96      0.95       181
           छ       0.89      0.99      0.94       181
           ज       0.97      0.99      0.98       180
         ज्ञ       1.00      1.00      1.00       180
           झ       0.98      0.98      0.98       182
           ञ       1.00      1.00      1.00       186
           ट       1.00      1.00      1.00       213
           ठ       1.00      1.00      1.00       214
           ड       1.00      1.00      1.00       215
           ढ       1.00      1.00      1.00       215
           ण      

In [ ]:
# Final training on full dataset
final_pca = PCA(n_components=0.95)
X_all_pca = final_pca.fit_transform(X_all_flat)

final_knn = KNeighborsClassifier(n_neighbors=n_neighbors, weights='distance')
final_knn.fit(X_all_pca, y_all_enc)

In [ ]:
import matplotlib.pyplot as plt
cm = confusion_matrix(all_y_true, all_y_pred)
fig, ax = plt.subplots(figsize=(12, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(ax=ax)
plt.show()

In [1]:
joblib.dump((final_knn), 'final_knn_model.pkl')
joblib.dump(final_pca, 'final_pca_model.pkl')
joblib.dump(le, "label_encoder.pkl")
print("knn and pca models saved")

NameError: name 'joblib' is not defined